# Week4_ex5 - Simple DC Machine

Magnetostatic model of a simple DC machine - rotated a 3-line coil through a full turn in 100 steps, solving at each angle to track torque, flux linkage, and the back-EMF it produces.

In [ ]:
import pandas as pd
import numpy as np
import os
import ansys.aedt.core
import math
import shutil
import matplotlib.pyplot as plt

In [ ]:
# 1. AEDT desktop interface
DT = ansys.aedt.core.Desktop(version="2025.2", non_graphical=False, student_version=True)

# 2. turn off autosave
DT.disable_autosave()

# 3. create the project and set the solution type
sol_type = "Magnetostatic"
M3D = ansys.aedt.core.maxwell.Maxwell3d(solution_type=sol_type)

# 4. odesign object, needed to reuse script-recording snippets
oDesign = M3D.odesign

In [ ]:
# 1. make a folder next to the script to hold the project
proj_name = "Week4_ex5"
dir = os.path.join(os.environ.get("ANSYS_PROJECT_DIR", os.getcwd()), proj_name)
print(dir)

os.makedirs(dir, exist_ok=True)

# 2. save the project
proj = M3D.oproject
proj.SaveAs(os.path.join(dir, f"{proj_name}.aedt"), True)

# 3. rename the design
desi_name = "Week4_ex5"
M3D.rename_design(desi_name, save=False)
M3D.save_project()

In [ ]:
# helper for finding coil terminal faces (kept from the source notebook - not actually used below)
# uses each face's center x-coordinate: the two faces furthest out on the x-axis are the terminals
def find_terminal_face(winding_obj) :
    terminal_face = []

    # find maximum x position of winding object
    max_x_pos = max(abs(face.center[0]) for face in winding_obj.faces)

    # append terminal face to array
    for face in winding_obj.faces :
        if abs(abs(face.center[0]) - max_x_pos) <= 0.0001 :
            terminal_face.append(face)

    # sort
    ter_out, ter_in = sorted(terminal_face, key=lambda x : x.center[0], reverse=True)

    return ter_out, ter_in

In [ ]:
line_length = 200
M3D["line_length"] = f"{line_length}mm"

line_diameter = 2
M3D["line_diameter"] = f"{line_diameter}mm"

line_num_seg = 12
M3D["line_num_seg"] = f"{line_num_seg}"

turns = 10     # number of coil turns
M3D["turns"] = f"{turns}"

Current = 10
M3D["Current"] = f"{Current}A"

mu0 = 4*math.pi*1e-7    # permeability of free space
B_field = 1
H_field = B_field/mu0
M3D["H_field"] = f"{H_field}"

elec_freq = 10   # electrical frequency
M3D["elec_freq"] = f"{elec_freq}Hz"

In [ ]:
# 1. create the setup, using the same Magnetostatic setup style used elsewhere in this notebook
my_setup = M3D.create_setup(name="Setup1")
my_setup.props["MaximumPasses"] = 20
my_setup.props["MinimumPasses"] = 2
my_setup.props["MinimumConvergedPasses"] = 1
my_setup.props["PercentRefinement"] = 30
my_setup.props["SolveFieldOnly"] = False
my_setup.props["PercentError"] = 1
my_setup.props["SolveMatrixAtLast"] = True
my_setup.props["UseNonLinearIterNum"] = False
my_setup.props["CacheSaveKind"] = "Delta"
my_setup.props["ConstantDelta"] = "0s"
my_setup.props["UseIterativeSolver"] = False
my_setup.props["RelativeResidual"] = 1E-06
my_setup.props["NonLinearResidual"] = 0.001
my_setup.props["RelaxationFactor"] = 1
my_setup.props["SmoothBHCurve"] = False
my_setup.props["MuOption"] = {"MuNonLinearBH": True}  # verify locally - same nested option as the raw InsertSetup version
my_setup.update()

In [ ]:
# 1. build the coil out of 4 straight segments
point_tmp = []
point_tmp.append(["line_length/2", "line_length/5*6", 0])
point_tmp.append(["line_length/2", "line_length/5", 0])
line_1 = M3D.modeler.create_polyline(points=point_tmp, name="line_1", material="copper",
                            xsection_type="Circle", xsection_width=line_diameter, xsection_num_seg=line_num_seg)

point_tmp = []
point_tmp.append(["-line_length/2", "line_length/5*6", 0])
point_tmp.append(["-line_length/2", "line_length/5", 0])
line_2 = M3D.modeler.create_polyline(points=point_tmp, name="line_2", material="copper",
                            xsection_type="Circle", xsection_width=line_diameter, xsection_num_seg=line_num_seg)

point_tmp = []
point_tmp.append(["-line_length/2", "line_length/5*6", 0])
point_tmp.append(["line_length/2", "line_length/5*6", 0])
line_3 = M3D.modeler.create_polyline(points=point_tmp, name="line_3", material="copper",
                            xsection_type="Circle", xsection_width=line_diameter, xsection_num_seg=line_num_seg)

point_tmp = []
point_tmp.append(["line_length/2", "line_length/5", 0])
point_tmp.append(["-line_length/2", "line_length/5", 0])
line_4 = M3D.modeler.create_polyline(points=point_tmp, name="line_4", material="copper",
                            xsection_type="Circle", xsection_width=line_diameter, xsection_num_seg=line_num_seg)


M3D.modeler.subtract(blank_list=[line_4], tool_list=[line_1], keep_originals=True)
M3D.modeler.subtract(blank_list=[line_3], tool_list=[line_1], keep_originals=True)
M3D.modeler.subtract(blank_list=[line_4], tool_list=[line_2], keep_originals=True)
M3D.modeler.subtract(blank_list=[line_3], tool_list=[line_2], keep_originals=True)

vector = [0, "-line_length/2-line_length/5", 0]
M3D.modeler.move(assignment=[line_1, line_2, line_3, line_4], vector=vector)


# 2. create the region
pad_value = ["line_length*6", "-line_length*6", "line_length*6", "-line_length*6", "line_length*6", "-line_length*6"]
region = M3D.modeler.create_region(pad_value=pad_value, pad_type='Absolute Position', name='Region')


# 3. dummy object, just to get a finer mesh where it matters
dummy = M3D.modeler.create_polyhedron(orientation="Y", center=(0.0, "-line_length/2*1.5", 0.0), origin=("-line_length/2*1.5", "-line_length/2*1.5", 0.0), height="line_length/2*3", 
                                      num_sides=36, name="dummy", material=None)

In [ ]:
tmp = [region.top_face_y, region.bottom_face_y, region.top_face_z, region.bottom_face_z]    
for f in tmp : 
    origin = f.center
    u_pos = origin.copy()
    u_pos[0] = u_pos[0] + 10
    M3D.assign_tangential_h_field(assignment=f, x_component_real=H_field, coordinate_system='Global', origin=origin, u_pos=u_pos)


# assign_zero_tangential_h_field has issues, so this uses the raw recorded script instead
oModule = oDesign.GetModule("BoundarySetup")

oModule.AssignZeroTangentialHField(
	[
		"NAME:ZeroTangentialHField1",
		"Faces:="		, [region.top_face_x.id, region.bottom_face_x.id]
	])

In [ ]:
M3D.assign_torque(assignment=line_1, coordinate_system='Global', is_positive=True, is_virtual=False, axis='Y', torque_name="torque_1")

M3D.assign_torque(assignment=line_2, coordinate_system='Global', is_positive=True, is_virtual=False, axis='Y', torque_name="torque_2")

M3D.assign_torque(assignment=line_3, coordinate_system='Global', is_positive=True, is_virtual=False, axis='Y', torque_name="torque_3")

In [ ]:
# 1. set the coil terminal on the 3D object
M3D.modeler.section(assignment=line_4, plane="YZ", create_new=True, section_cross_object=False)
coil_ter = M3D.modeler.sheet_objects[-1]

coil_terminal = M3D.assign_coil(assignment=coil_ter, conductors_number="turns", polarity="Negative", name="coil_terminal")


# 2. create a winding and add the coil to it
coil_1 = M3D.assign_winding(assignment=None, winding_type="Current", is_solid=False, current="Current", 
                            resistance=0, inductance=0, voltage=0, parallel_branches=1, phase=0, name="coil_1")

M3D.add_winding_coils(assignment=coil_1.name, coils=[coil_terminal.name])

In [ ]:
# Magnetostatic auto-refines mesh during the solve, so this isn't strictly required,
# but it's tightened up here to get more accurate results in the areas that matter
oModule = oDesign.GetModule("MeshSetup")

oModule.AssignLengthOp(
	[
		"NAME:Length1",
		"RefineInside:="	, True,
		"Enabled:="		, True,
		"Objects:="		, ["line_1","line_2","line_3","line_4"],
		"RestrictElem:="	, False,
		"NumMaxElem:="		, "1000",
		"RestrictLength:="	, True,
		"MaxLength:="		, "line_length/20"
	])

oModule.AssignLengthOp(
	[
		"NAME:Length2",
		"RefineInside:="	, True,
		"Enabled:="		, True,
		"Objects:="		, ["dummy"],
		"RestrictElem:="	, False,
		"NumMaxElem:="		, "1000",
		"RestrictLength:="	, True,
		"MaxLength:="		, "line_length/5"
	])

In [ ]:
## set up the data table ##

table_name = "data_table"

oModule = oDesign.GetModule("ReportSetup")

oModule.CreateReport(table_name, "Magnetostatic", "Data Table", "Setup1 : LastAdaptive", [], 
	[
		"line_length:="		, ["All"],
		"line_diameter:="	, ["Nominal"],
		"line_num_seg:="	, ["Nominal"],
		"Current:="		, ["Nominal"],
		"H_field:="		, ["Nominal"],
		"elec_freq:="		, ["Nominal"]
	], 
	[
		"X Component:="		, "line_length",
		"Y Component:="		, ["torque_1.Torque","torque_2.Torque","torque_3.Torque", "FluxLinkage(coil_1)"]
	])

In [ ]:
# 1. set the number of steps and the dataframe that will collect the results
step = 100
step_range = range(0, step)

pd_data = pd.DataFrame(columns=["angle [degree]", "torque line 1 [N.m]", "torque line 2 [N.m]", "torque line 3 [N.m]", "flux linkage [Wb]", "B-emf [V]"])


for i in step_range :

    # 2. solve
    my_setup.analyze()


    # 3. fix up the units on the data table (pasted from script recording)
    oModule = oDesign.GetModule("ReportSetup")
    oModule.ChangeProperty(
        [
            "NAME:AllTabs",
            [
                "NAME:Data Filter",
                [
                    "NAME:PropServers", 
                    f"{table_name}:torque_1.Torque:Curve1"
                ],
                [
                    "NAME:ChangedProps",
                    [
                        "NAME:Units",
                        "Value:="		, "NewtonMeter"
                    ]
                ]
            ]
        ])
    oModule.ChangeProperty(
        [
            "NAME:AllTabs",
            [
                "NAME:Data Filter",
                [
                    "NAME:PropServers", 
                    f"{table_name}:torque_2.Torque:Curve1"
                ],
                [
                    "NAME:ChangedProps",
                    [
                        "NAME:Units",
                        "Value:="		, "NewtonMeter"
                    ]
                ]
            ]
        ])
    oModule.ChangeProperty(
        [
            "NAME:AllTabs",
            [
                "NAME:Data Filter",
                [
                    "NAME:PropServers", 
                    f"{table_name}:torque_3.Torque:Curve1"
                ],
                [
                    "NAME:ChangedProps",
                    [
                        "NAME:Units",
                        "Value:="		, "NewtonMeter"
                    ]
                ]
            ]
        ])
    oModule.ChangeProperty(
        [
            "NAME:AllTabs",
            [
                "NAME:Data Filter",
                [
                    "NAME:PropServers", 
                    f"{table_name}:FluxLinkage(coil_1):Curve1"
                ],
                [
                    "NAME:ChangedProps",
                    [
                        "NAME:Units",
                        "Value:="		, "Wb"
                    ]
                ]
            ]
        ])


    # 4. export the data table and load it back in with pandas
    csv_name = "data_1.csv"
    dir_csv = os.path.join(dir, csv_name)

    if os.path.exists(dir_csv):        # remove the old csv if it's still there
        os.remove(dir_csv)
        print(f"removed old {csv_name}")
    else:
        print(f"{csv_name} not found, nothing to remove")

    table_name = "data_table"
    oModule = oDesign.GetModule("ReportSetup")
    oModule.ExportToFile(table_name, dir_csv, False)

    pd_tmp = pd.read_csv(dir_csv)  # load the exported row back in

    pd_data.loc[i, "torque line 1 [N.m]"] = pd_tmp.iloc[0][pd_tmp.columns[1]]  # add it to row i of pd_data
    pd_data.loc[i, "torque line 2 [N.m]"] = pd_tmp.iloc[0][pd_tmp.columns[2]]
    pd_data.loc[i, "torque line 3 [N.m]"] = pd_tmp.iloc[0][pd_tmp.columns[3]]
    pd_data.loc[i, "flux linkage [Wb]"] = pd_tmp.iloc[0][pd_tmp.columns[4]]
    pd_data.loc[i, "angle [degree]"] = i * 360 / step

    # 5. compute the back-EMF
    if i == 0:
        pd_data.loc[i, "B-emf [V]"] = 0
    else :
        pd_data.loc[i, "B-emf [V]"] = -( pd_data.loc[i, "flux linkage [Wb]"]-pd_data.loc[i-1, "flux linkage [Wb]"] )/(1/elec_freq/step)


    # 6. rotate one step
    M3D.modeler.rotate(assignment=[line_1, line_2, line_3, line_4, coil_ter], axis="Y", angle=360/step, units='deg')
    print(f"rotation {i+1}/{step} done")

    if i%(step//5) == 0 :
        display(pd_data)


pd_data.loc[0, "B-emf [V]"] = pd_data.loc[2, "B-emf [V]"]   # the first B-emf value is always 0, so back-fill it from the 3rd sample
    
my_setup.analyze()

In [ ]:
# reuse a cached run if one exists, so tweaking the plot below doesn't require redoing
# the whole 100-step sweep every time. Falls back to the sweep's own result if no cache is there yet.
# once you're happy with a run, copy Week4_ex5_result.csv to dir/output_success.csv to cache it.
cache_path = os.path.join(dir, "output_success.csv")
if os.path.exists(cache_path):
    pd_data = pd.read_csv(cache_path)
    print(f"loaded cached results from {cache_path}")
else:
    print("no cached results found - using the sweep's live results")

In [ ]:
## build the plots ##

# total torque
pd_data["total torque [N.m]"] = pd_data["torque line 1 [N.m]"] + pd_data["torque line 2 [N.m]"] + pd_data["torque line 3 [N.m]"]

# torque plot
plt.figure(figsize=(12, 8))

# column 0 on the x-axis, columns 1-3 on the y-axis
plt.plot(pd_data["angle [degree]"], pd_data["torque line 1 [N.m]"],linestyle='-', linewidth=4, markersize=6, label="Torque Line 1 [N.m]")
plt.plot(pd_data["angle [degree]"], pd_data["torque line 2 [N.m]"], linestyle='--', linewidth=4, markersize=6, label="Torque Line 2 [N.m]")
plt.plot(pd_data["angle [degree]"], pd_data["torque line 3 [N.m]"], linestyle='-', linewidth=2, markersize=6, label="Torque Line 3 [N.m]")
plt.plot(pd_data["angle [degree]"], pd_data["total torque [N.m]"], marker='o', linestyle='-', linewidth=4, markersize=6, label="Total Torque [N.m]", color='black')

# title and axis labels
plt.title("Torque vs Angle", fontsize=16)
plt.xlabel("Angle [degree]", fontsize=14)
plt.ylabel("Torque [N.m]", fontsize=14)

# annotate max/min for each series
for col in ["torque line 1 [N.m]", "torque line 3 [N.m]", "total torque [N.m]"]:
    max_val = pd_data[col].max()
    min_val = pd_data[col].min()
    max_angle = pd_data[pd_data[col] == max_val]["angle [degree]"].values[0]
    min_angle = pd_data[pd_data[col] == min_val]["angle [degree]"].values[0]
    plt.annotate(f'Max: {max_val:.3f}', xy=(max_angle, max_val), xytext=(max_angle, max_val + 0.5),
                 arrowprops=dict(facecolor='black', shrink=0.05), fontsize=10, color='black')
    plt.annotate(f'Min: {min_val:.3f}', xy=(min_angle, min_val), xytext=(min_angle, min_val - 0.5),
                 arrowprops=dict(facecolor='black', shrink=0.05), fontsize=10, color='black')

# legend
plt.legend(fontsize=12)

# grid
plt.grid(True, linestyle='--', linewidth=0.5)

# save + show
os.makedirs("Images", exist_ok=True)
plt.savefig("Images/Week4_ex5_torque.png")
plt.show()


# flux linkage and back-EMF plot
fig, ax1 = plt.subplots(figsize=(12, 8))

# first y-axis: flux linkage
ax1.set_xlabel("Angle [degree]", fontsize=14)
ax1.set_ylabel("Flux Linkage [Wb]", fontsize=14, color='tab:blue')
ax1.plot(pd_data["angle [degree]"], pd_data["flux linkage [Wb]"], marker='o', linestyle='-', linewidth=2, markersize=6, label="Flux Linkage [Wb]", color='tab:blue')
ax1.tick_params(axis='y', labelcolor='tab:blue')

# annotate max/min for flux linkage
flux_max_val = pd_data["flux linkage [Wb]"].max()
flux_min_val = pd_data["flux linkage [Wb]"].min()
flux_max_angle = pd_data[pd_data["flux linkage [Wb]"] == flux_max_val]["angle [degree]"].values[0]
flux_min_angle = pd_data[pd_data["flux linkage [Wb]"] == flux_min_val]["angle [degree]"].values[0]
ax1.annotate(f'Max: {flux_max_val:.2f}', xy=(flux_max_angle, flux_max_val), xytext=(flux_max_angle, flux_max_val + 0.1),
             arrowprops=dict(facecolor='blue', shrink=0.05), fontsize=10, color='blue')
ax1.annotate(f'Min: {flux_min_val:.2f}', xy=(flux_min_angle, flux_min_val), xytext=(flux_min_angle, flux_min_val - 0.1),
             arrowprops=dict(facecolor='blue', shrink=0.05), fontsize=10, color='blue')

# second y-axis: back-EMF
ax2 = ax1.twinx()
ax2.set_ylabel("B-emf [V]", fontsize=14, color='tab:red')
ax2.plot(pd_data["angle [degree]"], pd_data["B-emf [V]"], marker='s', linestyle='-', linewidth=2, markersize=6, label="B-emf [V]", color='tab:red')
ax2.tick_params(axis='y', labelcolor='tab:red')

# annotate max/min for back-EMF
bemf_max_val = pd_data["B-emf [V]"].max()
bemf_min_val = pd_data["B-emf [V]"].min()
bemf_max_angle = pd_data[pd_data["B-emf [V]"] == bemf_max_val]["angle [degree]"].values[0]
bemf_min_angle = pd_data[pd_data["B-emf [V]"] == bemf_min_val]["angle [degree]"].values[0]
ax2.annotate(f'Max: {bemf_max_val:.3f}', xy=(bemf_max_angle, bemf_max_val), xytext=(bemf_max_angle, bemf_max_val - 15),
             arrowprops=dict(facecolor='red', shrink=0.05), fontsize=10, color='red')
ax2.annotate(f'Min: {bemf_min_val:.3f}', xy=(bemf_min_angle, bemf_min_val), xytext=(bemf_min_angle, bemf_min_val + 15),
             arrowprops=dict(facecolor='red', shrink=0.05), fontsize=10, color='red')

# title
plt.title("Flux Linkage and B-emf vs Angle", fontsize=16)

# legend
fig.tight_layout()
fig.legend(loc="upper right", bbox_to_anchor=(1,1), bbox_transform=ax1.transAxes)

# grid
ax1.grid(True, linestyle='--', linewidth=0.5)

# save + show
plt.savefig("Images/Week4_ex5_flux_bemf.png")
plt.show()

In [ ]:
# saved flat, next to the notebook -- this is the final combined result
# (torque/flux/back-EMF for all 100 rotation steps), not scratch data
pd_data.to_csv("Week4_ex5_result.csv", index=False)

In [ ]:
M3D.save_project()